# Tutorial: a parameter scan with a live view

A **scan** runs the same SRIM calculation for several values of its parameters: here the
energy of boron ions in silicon. Each point is one `TRIM` run, the points run on worker
processes, and the results form one table with a row per point.

Set up SRIM once with `pysrim2 install` and check it with `pysrim2 doctor` (see the
[installation guide](https://pysrim2.readthedocs.io/en/latest/installation.html)). The
notebook finds SRIM by itself, natively or in the container; no path is needed.
SRIM-2013 only, **not** 2018. SRIM is (c) SRIM.com (James F. Ziegler), free for
non-commercial use only; see `LEGAL.rtf` next to your SRIM copy.

We will

- describe the base run and the parameter to vary,
- run the scan with `LiveView`, which shows the status and the plot while it runs,
- read the table and plot the mean depth against the energy.

The [scans page](https://pysrim2.readthedocs.io/en/latest/scans.html) has the rest:
grids and points, precision targets, the other extractors, the cache and the cluster
back end.

In [ ]:
from srim import TRIM, Ion, Layer, Target
from srim.scan import Scan, run_scan
from srim.ui import LiveView

## The base run

A scan starts from one run spec. `TRIM(...).to_spec()` gives it. The ion energy is
the parameter the scan changes, so its value here does not matter. `ranges=1` writes
`RANGE_3D.txt`, which the `implant_summary` extractor needs. The layer is 1000 nm of
silicon (`width` is in Angstrom), thick enough to stop every ion at these energies.

In [ ]:
target = Target([Layer.from_formula("Si", density=2.33, width=10000.0)])
base = TRIM(
    target,
    Ion("B", energy=20e3),  # eV
    number_ions=100,
    calculation=1,  # quick Kinchin-Pease
    ranges=1,
    random_seed=12345,
).to_spec()

## The scan

`grid` lists the values of each parameter; the points are all combinations. The keys are
dotted, as in `pysrim2 run --set`. `extract` names what to keep from each run (the
default is `implant_summary`); `ions` is the number of ions per point.

The scan folder is `scan-notebook`, relative to this notebook. `LiveView()` is the event
consumer that draws the status line and the plot in this cell, at most once a second.

In [ ]:
scan = Scan(
    base,
    grid={"ion.energy_keV": [10.0, 20.0, 30.0, 40.0]},
    extract=["implant_summary"],
    ions=100,
    seed=12345,  # the base seed: the same numbers on every run
)
result = run_scan(scan, workers=2, scan_dir="scan-notebook", on_event=LiveView())

## The table

One row per point: the scanned parameter, the extracted values (`mean_depth_A`,
`straggle_depth_A`, ...), `n_ions`, the seed, a `status` (`done` here) and the `path`
of the point's outputs. The same table is in `scan-notebook/result/scan.csv`.

In [ ]:
result.table[["ion.energy_keV", "status", "n_ions_stopped", "mean_depth_A", "straggle_depth_A"]]

## The plot

`ScanResult.plot(x, y)` draws the value `y` against the parameter `x`. Error bars come
from `<y>_err` when the scan has them (an estimator fills them in; a fixed ion count
does not). Mean depth in Angstrom.

In [ ]:
result.plot("ion.energy_keV", "mean_depth_A")

## Reruns and the cache

The runs of a scan are kept in the cache, `scan-notebook/cache` by default. Run the
same scan again (in the same scan folder) and no SRIM run is computed again: the
points have status `cached`. A scan with a different folder reuses nothing unless it
names the same cache (`cache=` in `run_scan`, or the `PYSRIM2_CACHE` variable).

Scans from the command line (`pysrim2 scan run`), on a cluster (`pysrim2 scan prepare`
and `collect`), and the live terminal views are described on the scans page.